# Обучение Transformer-агента на Мульти-Ассет данных (BTC + ETH)

В этом ноутбуке мы тестируем сразу два крупных нововведения:
1. **`load_multi_crypto_data`**: Загружает данные сразу по нескольким монетам без дублирования кода.
2. **Multi-Asset `TradingEnvV5`**: Среда теперь принимает список датафреймов и при каждом `reset()` случайно выбирает монету.
3. **`GatedTransformerExtractor`**: Новая архитектура сети, которая обрабатывает `n_stack` свечей через механизм Self-Attention.

In [8]:
import pandas as pd
import numpy as np
import os
import wandb

from core.data.data_loader import load_multi_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from agents.ppo_agent import create_ppo_agent
from core.experiment_manager import get_experiment_paths
from agents.callbacks import TradingMetricsCallback
from wandb.integration.sb3 import WandbCallback

In [9]:
# 1. Настройка эксперимента
exp_name = "ppo_transformer_multi_asset"
model_path, norm_path, tb_log_dir = get_experiment_paths(exp_name)

print(f"Model will be saved to: {model_path}")
print(f"Normalizer will be saved to: {norm_path}")
print(f"Tensorboard logs: {tb_log_dir}")

Model will be saved to: /Users/maximsinyaev/projects/trading_rl/trading_rl/models/experiments/ppo_transformer_multi_asset/ppo_model
Normalizer will be saved to: /Users/maximsinyaev/projects/trading_rl/trading_rl/models/experiments/ppo_transformer_multi_asset/vec_normalize.pkl
Tensorboard logs: /Users/maximsinyaev/projects/trading_rl/trading_rl/tensorboard_logs/ppo_transformer_multi_asset


In [10]:
# 2. Загрузка мульти-ассет данных
dfs_dict = load_multi_crypto_data(symbols=["BTCUSDT", "ETHUSDT"], start_date="2023-01-01", interval="4h")


🔄 Processing BTCUSDT...
✅ All data in cache, loading from disk...

🔄 Processing ETHUSDT...
✅ All data in cache, loading from disk...


In [11]:
# 3. Генерация фичей
fg = FeatureGenerator()

processed_dfs = []
for sym, df in dfs_dict.items():
    print(f"Генерация фичей для {sym}...")
    proc_df = fg.transform(df)
    processed_dfs.append(proc_df)
    print(f"{sym}: {len(proc_df)} свечей готовы.")

✅ HMM Model loaded from /Users/maximsinyaev/projects/trading_rl/trading_rl/models/hmm_model.pkl
Генерация фичей для BTCUSDT...
BTCUSDT: 7532 свечей готовы.
Генерация фичей для ETHUSDT...
ETHUSDT: 7532 свечей готовы.


In [12]:
# 4. Инициализация мульти-ассет среды
train_env = TradingEnvV5(df=processed_dfs, continuous_action=True, domain_randomization=True)

vec_env = DummyVecEnv([lambda: train_env])
vec_env = VecFrameStack(vec_env, n_stack=10)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

In [13]:
# 5. Инициализация Weights & Biases
run = wandb.init(
    project="trading_rl",
    name=exp_name,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/maximsinyaev/.netrc.
wandb: Currently logged in as: bemaxradio (bemaxradio-sinyaev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [agents] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


wandb: WARNING Artifact 
"source-trading_rl-_Users_maximsinyaev_projects_trading_rl_trading_rl_notebooks_12._train_transformer_multi_asset.i
pynb" already exists with the same content. No new version will be created.

In [14]:
# 6. Инициализация Трансформера
model = create_ppo_agent(
    vec_env,
    extractor_type="transformer",
    tensorboard_log=tb_log_dir
)

🚀 Initializing PPO (TRANSFORMER) on device: mps


/Users/maximsinyaev/projects/trading_rl/trading_rl/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/Users/maximsinyaev/projects/trading_rl/trading_rl/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [15]:
# 7. Обучение
callbacks = [
    TradingMetricsCallback(),
    WandbCallback(
        gradient_save_freq=50000,
        model_save_path=model_path,
        verbose=0,
    )
]

print(f"Запуск обучения {exp_name}...")
model.learn(total_timesteps=10000, callback=callbacks, progress_bar=True)

# 8. Сохранение результатов в папку эксперимента
model.save(model_path)
vec_env.save(norm_path)
wandb.finish()
print(f"✅ Эксперимент {exp_name} успешно сохранен!")

wandb: WARNING Found log directory outside of given root_logdir, dropping given root_logdir for event file in /Users/maximsinyaev/projects/trading_rl/trading_rl/tensorboard_logs/ppo_transformer_multi_asset/PPO_1


Output()

Запуск обучения ppo_transformer_multi_asset...


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.

env/mean_pnl,▆█▁█▂
env/mean_real_reward,▁▁▃▃▆▆████
global_step,▁▁▃▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆██████████
time/fps,█▁▁▁▂
train/approx_kl,▃▁▇█
train/clip_fraction,▃▁▅█
train/clip_range,▁▁▁▁
train/entropy_loss,▆█▁▁
train/explained_variance,▁▃▅█
train/learning_rate,▁▁▁▁
+4,...


✅ Эксперимент ppo_transformer_multi_asset успешно сохранен!
